# tool_choice的使用

bind_tools 可以传递参数 tool_choice ，用于控制是否强制使用工具。

该字段最终会作为 payload 的 tool_choice 字段传递给模型，OpenAI和Deepseek的官方API服务对于
tool_choice 的取值做了相同的规定。

none ：模型不会调用任何工具。

auto ： 默认值 ，模型可以自主决定不调用或调用任意数量的工具。

required ：模型必须调用工具，数量不限。

此外， tool_choice 还支持传递 any ，等价于 required 。


## 1、tool_choice取值为none

In [12]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os


# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    extra_body={"enable_thinking": False},
    # max_tokens=10,
)

In [4]:
from langchain.tools import tool
# from langchain.messages import HumanMessage
from rich import print as r

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
         city: 城市名称
    """
    return f"{city}天气晴朗~~~"

model_with_tools = model.bind_tools([get_weather], tool_choice="none")

response = model_with_tools.invoke("北京今天的天气如何呢？")
# response.pretty_print()
r(response)

AIMessage(
    content='抱歉，作为人工智能，我无法实时联网获取今天最新的天气数据。\n\n建议您直接查看手机上的天气APP，或者在搜
索引擎中输入“北京今天天气”，以获取最准确的实时气温、降水概率以及空气质量（AQI）等信息。\n\n如果您正在为近期的北京之
行做准备，可以告诉我您计划出行的月份，我可以为您提供北京该季节的气候特点、穿衣指南以及旅游建议！',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 512,
            'prompt_tokens': 16,
            'total_tokens': 528,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 428,
                'rejected_prediction_tokens': None,
                'text_tokens': 512
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 16}
        },
        'model_provider': 'openai',
        'model_name': 'qwen3.7-plus',
        'system_fingerprint': None,
        'id': 'chatcmpl-076c009f-55d1-9b73-b9e4-79b1ae6c1150',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f51a5-6867-7981-9b36-d768c58b03c2-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 16,
        'output_tokens': 512,
        'total_tokens': 528,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 428}
    }
)

## 2、tool_choice取值为auto

In [6]:
from langchain.tools import tool
# from langchain.messages import HumanMessage
from rich import print as r

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
         city: 城市名称
    """
    return f"{city}天气晴朗~~~"

model_with_tools = model.bind_tools([get_weather], tool_choice="auto")

response = model_with_tools.invoke("我是谁？")
# response.pretty_print()
r(response)

AIMessage(
    content='我不知道您是谁，因为我是一个人工智能助手，无法获取您的个人身份信息。不过，您可以把我当成您的专属助手，
有什么我可以帮您的吗？',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 285,
            'prompt_tokens': 275,
            'total_tokens': 560,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 250,
                'rejected_prediction_tokens': None,
                'text_tokens': 285
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 275}
        },
        'model_provider': 'openai',
        'model_name': 'qwen3.7-plus',
        'system_fingerprint': None,
        'id': 'chatcmpl-5cd0de0f-b83f-9a50-9cf1-6844e273a7d3',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f51aa-58e9-76f3-879f-621a5b8ac345-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 275,
        'output_tokens': 285,
        'total_tokens': 560,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 250}
    }
)

## 3、tool_choice取值为required

tool_choice 还支持传递 any ，等价于 required 。

开启 enable_thinking（深度思考模式）时，阿里云官方限制：tool_choice 只能用 "auto" / "none"；禁止使用 "required" 和对象格式强制指定工具，否则直接返回 400‑InvalidParameter 报错，和 DeepSeek‑V4 规则完全一致阿里云帮助...

所以在模型初始化的时候应该增加`extra_body={"enable_thinking": False},`


In [14]:
from langchain.tools import tool
# from langchain.messages import HumanMessage
from rich import print as r


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    获取当日天气

    Args:
         city: 城市名称
    """
    return f"{city}天气晴朗~~~"

model_with_tools = model.bind_tools([get_weather], tool_choice="required")

response = model_with_tools.invoke("我是谁？")
# response.pretty_print()
r(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 22,
            'prompt_tokens': 280,
            'total_tokens': 302,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 280}
        },
        'model_provider': 'openai',
        'model_name': 'qwen3.7-plus',
        'system_fingerprint': None,
        'id': 'chatcmpl-94def98b-5f4b-91b5-ba6c-711cd500569b',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019f51af-7b80-79f0-b405-92778aafc4ad-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_32656555d75041f8a28f4985',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 280,
        'output_tokens': 22,
        'total_tokens': 302,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {}
    }
)

## 4、强制调用指定的工具

In [25]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage


@tool(parse_docstring=True)
def get_weather1(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天晴朗'

@tool(parse_docstring=True)
def get_weather2(city: str) -> str:
    """
    获取当日天气

    Args:
        city: 城市名称
    """
    return f'{city}当天下雨'

# tool_choice="get_weather2" 强制模型只能选择get_weather2工具
model_with_tools = model.bind_tools([get_weather1, get_weather2], tool_choice="get_weather1")
messages = [
    HumanMessage(content="杭州今天天气如何？")
]
response = model_with_tools.invoke(messages)
response.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_weather1 (call_0a2e774915f342529658ffc7)
 Call ID: call_0a2e774915f342529658ffc7
  Args:
    city: 杭州
